In [1]:
!pip install fair-esm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 3.4 MB/s eta 0:00:00


In [2]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

In [3]:
# Dataset Class with data augmentation
class PeptideDataset(Dataset):
    def __init__(self, csv_file, augment=False):
        df = pd.read_csv(csv_file)
        self.sequences = df['sequence'].astype(str).tolist()
        self.labels = df['FRS'].tolist()
        self.augment = augment

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]

        # Simple data augmentation - reverse sequence
        if self.augment and torch.rand(1).item() > 0.5:
            seq = seq[::-1]

        return seq, label

In [4]:
# Main Execution
if __name__ == "__main__":
    # Initialize with data augmentation for training
    full_dataset = PeptideDataset("01_AO_db_testsplit (1).csv", augment=True)

    # Stratified split
    from sklearn.model_selection import train_test_split
    indices = list(range(len(full_dataset)))
    labels = [full_dataset[i][1] for i in indices]
    train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=labels, random_state=42)

    train_dataset = torch.utils.data.Subset(full_dataset, train_idx)
    val_dataset = torch.utils.data.Subset(
        PeptideDataset("01_AO_db_testsplit (1).csv", augment=False),
        val_idx
    )

In [5]:
# Extract sequences and labels for train_dataset
train_sequences = [full_dataset.sequences[i] for i in train_idx]
train_labels = [full_dataset.labels[i] for i in train_idx]

# Create a DataFrame for the training set and save to CSV
train_df = pd.DataFrame({'sequence': train_sequences, 'FRS': train_labels})
train_df.to_csv('train_dataset.csv', index=False)
print("train_dataset.csv created successfully.")

# Extract sequences and labels for val_dataset
# Note: Assuming val_dataset is also derived from the same full_dataset as train_dataset
val_sequences = [full_dataset.sequences[i] for i in val_idx]
val_labels = [full_dataset.labels[i] for i in val_idx]

# Create a DataFrame for the validation set and save to CSV
val_df = pd.DataFrame({'sequence': val_sequences, 'FRS': val_labels})
val_df.to_csv('val_dataset.csv', index=False)
print("val_dataset.csv created successfully.")

train_dataset.csv created successfully.
val_dataset.csv created successfully.


In [6]:
train = pd.read_csv("train_dataset.csv")
val = pd.read_csv("val_dataset.csv")

train["sequence"] = train["sequence"].str.strip().str.upper()
val["sequence"] = val["sequence"].str.strip().str.upper()

print("Train size:", len(train))
print("Validation size:", len(val))

train_set = set(train['sequence'])
val_set = set(val['sequence'])

exact_overlap = train_set.intersection(val_set)

print("Exact duplicate sequences:", len(exact_overlap))
print("Leakage rate:", len(exact_overlap)/len(val))

Train size: 32000
Validation size: 8000
Exact duplicate sequences: 0
Leakage rate: 0.0


In [7]:
def write_fasta(df, filename):
    with open(filename, "w") as f:
        for i, seq in enumerate(df.sequence):
            f.write(f">seq_{i}\n{seq}\n")

write_fasta(train, "train.fasta")
write_fasta(val, "val.fasta")

In [8]:
import getpass
import os

password = getpass.getpass()  # This prompts for the password in the notebook output area or terminal
#command = "sudo -S apt-get update" # Replace 'apt-get update' with your desired command
command = "sudo -S apt-get install cd-hit"

# Use os.system to run the command
os.system(f'echo {password} | {command}')

··········


0

In [9]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.8 MB/s eta 0:00:00


In [10]:
!cat train.fasta val.fasta > combined.fasta
!cd-hit -i combined.fasta -o combined_90 -c 0.9 -n 5

Program: CD-HIT, V4.8.1 (+OpenMP), Aug 20 2021, 08:39:56
Command: cd-hit -i combined.fasta -o combined_90 -c 0.9 -n 5

Started: Mon Mar 30 22:26:49 2026
                            Output                              
----------------------------------------------------------------
total seq: 11376
longest and shortest : 44 and 11
Total letters: 168815
Sequences have been sorted

Approximated minimal memory consumption:
Sequence        : 1M
Buffer          : 1 X 10M = 10M
Table           : 1 X 65M = 65M
Miscellaneous   : 0M
Total           : 77M

Table limit with the given memory limit:
Max number of representatives: 4000000
Max number of word counting entries: 90265380

comparing sequences from          0  to      11376
..........    10000  finished       6809  clusters
.
    11376  finished       7753  clusters

Approximated maximum memory consumption: 78M
writing new database
writing clustering information
program completed !

Total CPU time 0.38


In [11]:
def parse_clusters(clstr_file):
    clusters = []
    current = []
    with open(clstr_file) as f:
        for line in f:
            if line.startswith(">Cluster"):
                if current:
                    clusters.append(current)
                current = []
            else:
                current.append(line.strip())
        clusters.append(current)
    return clusters

clusters = parse_clusters("combined_90.clstr")

leak_clusters = 0
for c in clusters:
    train_flag = any("train" in x for x in c)
    val_flag = any("val" in x for x in c)
    if train_flag and val_flag:
        leak_clusters += 1

print("Clusters containing both train and val:", leak_clusters)

Clusters containing both train and val: 0


In [12]:
from itertools import product
import numpy as np

def kmers(seq, k=6):
    return set(seq[i:i+k] for i in range(len(seq)-k+1))

train_k = [kmers(s,6) for s in train.sequence]
val_k = [kmers(s,6) for s in val.sequence]

def jaccard(a,b):
    union_len = len(a | b)
    if union_len == 0:
        return 0.0  # Or handle as appropriate, e.g., NaN if truly undefined
    return len(a & b) / union_len

high_similarity_pairs = 0

for i, vk in enumerate(val_k):
    for tk in train_k:
        if jaccard(vk, tk) > 0.8:
            high_similarity_pairs += 1
            break

print("Val sequences with high motif overlap:", high_similarity_pairs)

Val sequences with high motif overlap: 174


In [13]:
import torch
import esm
import numpy as np
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

device = "cpu"

model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
batch_converter = alphabet.get_batch_converter()

model = model.to(device)
model.eval()

############################################################
# Memory-safe embedding computation
############################################################

def compute_embeddings(seqs, batch_size=32):

    embeddings = []

    for i in tqdm(range(0, len(seqs), batch_size)):

        batch_seqs = seqs[i:i+batch_size]
        data = [(str(j), s) for j,s in enumerate(batch_seqs)]

        _, _, tokens = batch_converter(data)
        tokens = tokens.to(device)

        with torch.no_grad():
            out = model(tokens, repr_layers=[6])

        reps = out["representations"][6]

        for j, seq in enumerate(batch_seqs):
            emb = reps[j, 1:len(seq)+1].mean(0).cpu().numpy()
            embeddings.append(emb)

        del tokens, out, reps
        torch.cuda.empty_cache()

    return np.vstack(embeddings)

############################################################
# Compute embeddings safely
############################################################

train_emb = compute_embeddings(train.sequence.tolist(), batch_size=32)
val_emb = compute_embeddings(val.sequence.tolist(), batch_size=32)

print("Train embeddings:", train_emb.shape)
print("Val embeddings:", val_emb.shape)

############################################################
# Chunked cosine similarity (avoids NxM matrix)
############################################################

threshold = 0.95
semantic_leak = 0

chunk_size = 200

for i in tqdm(range(0, len(val_emb), chunk_size)):

    val_chunk = val_emb[i:i+chunk_size]

    sims = cosine_similarity(val_chunk, train_emb)

    max_sim = sims.max(axis=1)

    semantic_leak += np.sum(max_sim > threshold)

    del sims

print("Val sequences with cosine similarity > 0.95:", semantic_leak)

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t6_8M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t6_8M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D-contact-regression.pt


100%|██████████| 250/250 [01:26<00:00,  2.89it/s]


Train embeddings: (32000, 320)
Val embeddings: (8000, 320)


100%|██████████| 40/40 [00:06<00:00,  6.32it/s]

Val sequences with cosine similarity > 0.95: 7986


In [14]:
from sklearn.neighbors import NearestNeighbors

combined_emb = np.vstack([train_emb, val_emb])
labels = ["train"]*len(train_emb) + ["val"]*len(val_emb)

nbrs = NearestNeighbors(n_neighbors=5, metric="cosine").fit(combined_emb)
distances, indices = nbrs.kneighbors(combined_emb)

cross_neighbors = 0
for i, neigh in enumerate(indices):
    for n in neigh[1:]:
        if labels[i] != labels[n]:
            cross_neighbors += 1
            break

print("Sequences with cross-split nearest neighbors:", cross_neighbors)

Sequences with cross-split nearest neighbors: 27060


In [16]:
print("\n==== DATA LEAKAGE SUMMARY ====")
print("Exact duplicates:", len(exact_overlap))
print("CD-HIT cluster leakage:", leak_clusters)
print("High k-mer leakage:", high_similarity_pairs)
print("High semantic leakage:", semantic_leak)
print("Cross-nearest-neighbor leakage:", cross_neighbors)


==== DATA LEAKAGE SUMMARY ====
Exact duplicates: 0
CD-HIT cluster leakage: 0
High k-mer leakage: 174
High semantic leakage: 7986
Cross-nearest-neighbor leakage: 27060


In [17]:
from itertools import product
import numpy as np
import pandas as pd

def kmers(seq, k=6):
    return set(seq[i:i+k] for i in range(len(seq)-k+1))

def jaccard(a,b):
    union_len = len(a | b)
    if union_len == 0:
        return 0.0
    return len(a & b) / union_len

def remove_high_kmer_leakage(train_df, val_df, k=6, threshold=0.8):
    """
    Removes sequences from the validation set that have high k-mer similarity
    with any sequence in the training set.

    Args:
        train_df (pd.DataFrame): DataFrame containing training sequences.
        val_df (pd.DataFrame): DataFrame containing validation sequences.
        k (int): Length of k-mer to use for similarity calculation.
        threshold (float): Jaccard similarity threshold.

    Returns:
        pd.DataFrame: Filtered validation DataFrame.
    """
    print(f"Initial validation size: {len(val_df)}")

    train_k_sets = [kmers(s, k) for s in train_df.sequence]
    val_k_sets = [kmers(s, k) for s in val_df.sequence]

    val_to_keep_indices = []
    removed_count = 0

    for i, vk in enumerate(val_k_sets):
        has_high_similarity = False
        for tk in train_k_sets:
            if jaccard(vk, tk) > threshold:
                has_high_similarity = True
                break
        if not has_high_similarity:
            val_to_keep_indices.append(i)
        else:
            removed_count += 1

    filtered_val_df = val_df.iloc[val_to_keep_indices].copy()
    print(f"Removed {removed_count} sequences from validation set due to high k-mer similarity.")
    print(f"New validation size: {len(filtered_val_df)}")

    return filtered_val_df

# Assuming 'train' and 'val' DataFrames are already loaded from previous cells
val_filtered = remove_high_kmer_leakage(train, val)

# Re-calculate k-mer leakage with the filtered validation set
train_k_filtered = [kmers(s,6) for s in train.sequence]
val_k_filtered = [kmers(s,6) for s in val_filtered.sequence]

high_similarity_pairs_after_filter = 0

for i, vk in enumerate(val_k_filtered):
    for tk in train_k_filtered:
        if jaccard(vk, tk) > 0.8:
            high_similarity_pairs_after_filter += 1
            break

print("Val sequences with high motif overlap (after filtering):", high_similarity_pairs_after_filter)

Initial validation size: 8000
Removed 174 sequences from validation set due to high k-mer similarity.
New validation size: 7826
Val sequences with high motif overlap (after filtering): 0


In [18]:
# Save the DataFrames to CSV files
train_df.to_csv("train_filtered.csv", index=False)
val_filtered.to_csv("val_filtered.csv", index=False)

In [19]:
# Convert each set of k-mers into a space-separated string
train_k_strings = [' '.join(list(s)) for s in train_k_filtered]
val_k_strings = [' '.join(list(s)) for s in val_k_filtered]

# Create DataFrames from these lists of strings
train_k_df = pd.DataFrame({'k_mers': train_k_strings})
val_k_df = pd.DataFrame({'k_mers': val_k_strings})

# Save the DataFrames to CSV files
train_k_df.to_csv("train_k_filtered.csv", index=False)
val_k_df.to_csv("val_k_filtered.csv", index=False)

In [20]:
# Recompute embeddings for the k-mer filtered validation set
# Using the previously defined compute_embeddings function
val_emb_kmer_filtered = compute_embeddings(val_filtered.sequence.tolist(), batch_size=32)

print("Train embeddings shape:", train_emb.shape)
print("K-mer filtered validation embeddings shape:", val_emb_kmer_filtered.shape)

def remove_high_semantic_leakage(train_embeddings, val_embeddings, val_df, threshold=0.95, chunk_size=200):
    """
    Removes sequences from the validation set that have high semantic similarity
    with any sequence in the training set based on ESM embeddings.

    Args:
        train_embeddings (np.ndarray): Embeddings for the training set.
        val_embeddings (np.ndarray): Embeddings for the validation set (to be filtered).
        val_df (pd.DataFrame): DataFrame corresponding to val_embeddings.
        threshold (float): Cosine similarity threshold.
        chunk_size (int): Size of chunks for cosine similarity calculation.

    Returns:
        pd.DataFrame: Semantically filtered validation DataFrame.
    """
    print(f"Initial validation size (semantic filter): {len(val_df)}")

    val_to_keep_indices = []
    removed_count = 0

    # Process validation embeddings in chunks
    for i in tqdm(range(0, len(val_embeddings), chunk_size), desc="Semantic filtering"):
        val_chunk = val_embeddings[i:i+chunk_size]

        # Calculate cosine similarity between val_chunk and all train_embeddings
        sims = cosine_similarity(val_chunk, train_embeddings)

        # Find the maximum similarity for each validation sequence in the chunk
        max_sims_in_chunk = sims.max(axis=1)

        # Determine which sequences in the chunk to keep
        for j, max_s in enumerate(max_sims_in_chunk):
            original_idx_in_val_df = i + j
            if max_s <= threshold:
                val_to_keep_indices.append(original_idx_in_val_df)
            else:
                removed_count += 1
        del sims

    filtered_val_df = val_df.iloc[val_to_keep_indices].copy()
    print(f"Removed {removed_count} sequences from validation set due to high semantic similarity (>{threshold}).")
    print(f"New validation size (after semantic filtering): {len(filtered_val_df)}")

    return filtered_val_df

# Apply semantic leakage removal
val_semantic_filtered = remove_high_semantic_leakage(train_emb, val_emb_kmer_filtered, val_filtered, threshold=0.95)

# Save the datasets
train.to_csv('train_semantic_filtered.csv', index=False)
val_semantic_filtered.to_csv('val_semantic_filtered.csv', index=False)

print("\n'train_semantic_filtered.csv' and 'val_semantic_filtered.csv' created successfully.")

100%|██████████| 245/245 [01:29<00:00,  2.75it/s]


Train embeddings shape: (32000, 320)
K-mer filtered validation embeddings shape: (7826, 320)
Initial validation size (semantic filter): 7826


Semantic filtering: 100%|██████████| 40/40 [00:05<00:00,  6.83it/s]


Removed 7812 sequences from validation set due to high semantic similarity (>0.95).
New validation size (after semantic filtering): 14

'train_semantic_filtered.csv' and 'val_semantic_filtered.csv' created successfully.


In [21]:
from sklearn.neighbors import NearestNeighbors
import numpy as np
import pandas as pd

# Recompute embeddings for the semantically filtered validation set
# This ensures we are working with embeddings corresponding to the latest filtered sequences
val_emb_semantic_filtered_final = compute_embeddings(val_semantic_filtered.sequence.tolist(), batch_size=32)

print("Train embeddings shape:", train_emb.shape)
print("Semantic filtered validation embeddings shape:", val_emb_semantic_filtered_final.shape)

def remove_cross_nn_leakage(train_embeddings, current_val_embeddings, current_val_df, n_neighbors=5):
    """
    Removes sequences from the validation set that have training sequences
    as one of their N nearest neighbors in the embedding space.

    Args:
        train_embeddings (np.ndarray): Embeddings for the training set.
        current_val_embeddings (np.ndarray): Embeddings for the current validation set (to be filtered).
        current_val_df (pd.DataFrame): DataFrame corresponding to current_val_embeddings.
        n_neighbors (int): Number of nearest neighbors to consider.

    Returns:
        pd.DataFrame: Validation DataFrame after removing cross-nearest-neighbor leakage.
    """
    print(f"Initial validation size (NN filter): {len(current_val_df)}")

    # Combine train and current validation embeddings
    combined_emb_local = np.vstack([train_embeddings, current_val_embeddings])
    # Create labels to distinguish train and val in the combined set
    labels_local = np.array(["train"] * len(train_embeddings) + ["val"] * len(current_val_embeddings))

    # Fit NearestNeighbors model on the combined embeddings
    nbrs_local = NearestNeighbors(n_neighbors=n_neighbors, metric="cosine").fit(combined_emb_local)

    val_to_keep_indices = []
    removed_count = 0

    # Iterate through each validation embedding
    for i in range(len(current_val_embeddings)):
        # Query for its neighbors within the combined set
        # Note: kneighbors returns distances and indices. We only need indices.
        _, neighbor_indices = nbrs_local.kneighbors(current_val_embeddings[i].reshape(1, -1))

        is_leaking = False
        # Check if any of the neighbors are from the training set
        # We iterate from the second neighbor if the first one is itself (which happens when n_neighbors > 0)
        # or iterate all if n_neighbors=1, as the point itself will be its closest neighbor.
        # If the point itself is in the validation set, its index will be >= len(train_embeddings).
        # We are looking for neighbors from the training set, i.e., index < len(train_embeddings).

        # Loop through all neighbors found (excluding self-match if present and belongs to val set)
        for neighbor_idx in neighbor_indices.flatten():
            # If the neighbor is a training sequence
            if neighbor_idx < len(train_embeddings):
                is_leaking = True
                break

        if not is_leaking:
            val_to_keep_indices.append(i)
        else:
            removed_count += 1

    filtered_val_df = current_val_df.iloc[val_to_keep_indices].copy()
    print(f"Removed {removed_count} sequences from validation set due to cross-nearest-neighbor leakage.")
    print(f"New validation size (after NN filtering): {len(filtered_val_df)}")

    return filtered_val_df

# Apply cross-nearest-neighbor leakage removal
val_neighbor_filtered = remove_cross_nn_leakage(train_emb, val_emb_semantic_filtered_final, val_semantic_filtered, n_neighbors=5)

# Save the datasets
train.to_csv('train_neighbor_filtered.csv', index=False)
val_neighbor_filtered.to_csv('val_neighbor_filtered.csv', index=False)

print("\n'train_neighbor_filtered.csv' and 'val_neighbor_filtered.csv' created successfully.")

100%|██████████| 1/1 [00:00<00:00,  5.25it/s]


Train embeddings shape: (32000, 320)
Semantic filtered validation embeddings shape: (14, 320)
Initial validation size (NN filter): 14
Removed 14 sequences from validation set due to cross-nearest-neighbor leakage.
New validation size (after NN filtering): 0

'train_neighbor_filtered.csv' and 'val_neighbor_filtered.csv' created successfully.
